In [ ]:
import numpy as np
import pandas as pd
import cvxpy as cp
from scipy.spatial import KDTree
from scipy.interpolate import interp1d
import plotly.graph_objects as go
from scipy.interpolate import CubicSpline
from sklearn.metrics import root_mean_squared_error as RMSE
from Functions.Graphs import *
import optuna
from optuna.samplers import RandomSampler
from optuna.exceptions import TrialPruned

def SelSampler(mode='auto'):
    '''mode: auto, random, tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        sampler = optuna.samplers.TPESampler(
            multivariate=True, 
            n_startup_trials=10
        )
    elif mode == 'random':
        sampler = RandomSampler()
    return sampler

# --- Adaptive Turning Speed Function ---
def TurningSpeed(u_control_1, V_cruising, V_turning, k=0.15):

    V_turning = V_turning * 0.9
    angulo_abs = abs((u_control_1))

    # Calculate exponential decay component
    speed_drop = (V_cruising - V_turning) * np.exp(-k * angulo_abs)
    
    # Floor the output cleanly at V_turning
    V_ref = V_turning + speed_drop
    
    return float(V_ref)



In [ ]:
# --- NEW: Fully Vectorized Matrix Generation (Massive Speedup) ---
def get_linear_model_matrices_all(x_current, u_prev, dt, n_horizon, l=2.5):
    """
    Computes all linearization matrices across the entire horizon at once.
    Eliminates the internal Python loop.
    """
    # Assuming constant nominal trajectory over the prediction horizon for initialization
    v = np.full(n_horizon, x_current[2])
    psi = np.full(n_horizon, x_current[3])
    a = np.full(n_horizon, u_prev[0])
    delta = np.full(n_horizon, u_prev[1])
    
    beta = np.arctan(0.5 * np.tan(delta))
    cos_psi_beta = np.cos(psi + beta)
    sin_psi_beta = np.sin(psi + beta)
    
    A_list = np.zeros((n_horizon, 4, 4))
    B_list = np.zeros((n_horizon, 4, 2))
    c_list = np.zeros((n_horizon, 4))
    
    dbeta_ddelta = 0.5 * (1 + np.tan(delta)**2) / (1 + (0.5 * np.tan(delta))**2)
    dx_ddelta = -dt * v * sin_psi_beta * dbeta_ddelta
    dy_ddelta = dt * v * cos_psi_beta * dbeta_ddelta
    dpsi_ddelta = dt * (v / l) * np.cos(beta) * dbeta_ddelta
    
    for k in range(n_horizon):
        A_list[k] = [
            [1.0, 0.0, dt * cos_psi_beta[k], -dt * v[k] * sin_psi_beta[k]],
            [0.0, 1.0, dt * sin_psi_beta[k],  dt * v[k] * cos_psi_beta[k]],
            [0.0, 0.0, 1.0,                0.0],
            [0.0, 0.0, dt * np.sin(beta[k])/l, 1.0]
        ]
        B_list[k] = [
            [0.0, dx_ddelta[k]],
            [0.0, dy_ddelta[k]],
            [dt,  0.0],
            [0.0, dpsi_ddelta[k]]
        ]
        
        x_next_nominal = np.array([
            x_current[0] + dt * v[k] * cos_psi_beta[k],
            x_current[1] + dt * v[k] * sin_psi_beta[k],
            v[k] + dt * a[k],
            psi[k] + dt * (v[k] / l) * np.sin(beta[k])
        ])
        c_list[k] = x_next_nominal - A_list[k] @ x_current - B_list[k] @ u_prev
        
    return A_list, B_list, c_list

In [29]:
def SimulateRT(dt=0.25, n_horizon=20, sim_steps=500, track_percentual=1,
               W_X=1, W_Y=1, W_speed=10, W_acc=1.5, W_delta=0.25, W_U0=1, W_U1=2,
               size=1, show=False):
    
    BreakCheck = False
    path = r'DyntheticDataset\RaceTrack4.csv' 
    try:
        df = pd.read_csv(path)
        x_mid = df['x_coords'].values[:] * size
        y_mid = df['y_coords'].values[:] * size
    except FileNotFoundError:
        theta = np.linspace(0, 2*np.pi, 200)
        x_mid = 100 * np.cos(theta) + 100
        y_mid = 100 * np.sin(theta) + 100

    track_points = np.vstack((x_mid, y_mid)).T
    track_tree = KDTree(track_points)

    dx = np.diff(x_mid)
    dy = np.diff(y_mid)
    segment_lengths = np.sqrt(dx**2 + dy**2)
    s_coor = np.insert(np.cumsum(segment_lengths), 0, 0.0)
    track_length = s_coor[-1]
    get_x_at_s = interp1d(s_coor, x_mid, kind='linear', bounds_error=False, fill_value="extrapolate")
    get_y_at_s = interp1d(s_coor, y_mid, kind='linear', bounds_error=False, fill_value="extrapolate")

    V_cruising, V_turning = 16, 8
    l = 2.5 

    s_total_traveled = 0.0
    last_current_idx = 0
    x_current = np.array([x_mid[0], y_mid[0], 0.0, 0.0])
    u_prev = np.array([0.0, 0.0])

    t_history, x_history, y_history, v_history = [0.0], [x_current[0]], [x_current[1]], [x_current[2]]
    v_ref_history, acc_history, delta_history, psi_history = [0.0], [0.0], [0.0], [0.0]
    turning_history = [0]

    turning = False
    t_lim = 50 * track_percentual
    sim_steps = int(sim_steps * track_percentual * 1.1)

    # ==============================================================================
    # --- CVXPY PARAMETRIC SETUP (COMPILES ONLY ONCE PER SIMULATION) ---
    # ==============================================================================
    X_cvx = cp.Variable((4, n_horizon + 1))
    U_cvx = cp.Variable((2, n_horizon))
    
    x_init_param = cp.Parameter(4)
    u_prev_param = cp.Parameter(2)
    x_ref_param = cp.Parameter(n_horizon)
    y_ref_param = cp.Parameter(n_horizon)
    v_ref_param = cp.Parameter(n_horizon)
    W_acc_param = cp.Parameter(nonneg=True)
    
    A_params = [cp.Parameter((4, 4)) for _ in range(n_horizon)]
    B_params = [cp.Parameter((4, 2)) for _ in range(n_horizon)]
    c_params = [cp.Parameter(4) for _ in range(n_horizon)]
    
    cost = 0
    constraints = [X_cvx[:, 0] == x_init_param]
    
    for k in range(n_horizon):
        #print('horizon pred')
        constraints += [X_cvx[:, k+1] == A_params[k] @ X_cvx[:, k] + B_params[k] @ U_cvx[:, k] + c_params[k]]
        
        cost += W_X * cp.square(X_cvx[0, k] - x_ref_param[k])
        cost += W_Y * cp.square(X_cvx[1, k] - y_ref_param[k])
        cost += W_speed * cp.square(X_cvx[2, k] - v_ref_param[k])
        cost += W_acc * cp.square(U_cvx[0, k])
        cost += W_acc_param * cp.square(U_cvx[0, k])

        cost += W_delta * cp.square(U_cvx[1, k])
        
        if k == 0:
            cost += W_U0 * cp.square(U_cvx[0, 0] - u_prev_param[0])
            cost += W_U1 * cp.square(U_cvx[1, 0] - u_prev_param[1])
        else:
            cost += W_U0 * cp.square(U_cvx[0, k] - U_cvx[0, k-1])
            cost += W_U1 * cp.square(U_cvx[1, k] - U_cvx[1, k-1])
            
        constraints += [U_cvx[0, k] >= -7.0, U_cvx[0, k] <= 3.0]
        constraints += [U_cvx[1, k] >= -np.deg2rad(30), U_cvx[1, k] <= np.deg2rad(30)]
        constraints += [X_cvx[2, k] >= 0.0, X_cvx[2, k] <= V_cruising]
        
    prob = cp.Problem(cp.Minimize(cost), constraints)
    # ==============================================================================

    for step in range(sim_steps):
        if step % 50 == 0 and show: 
            print(f'Step: {step} | Speed: {x_current[2]:.2f} m/s | Distance traveled: {s_total_traveled:.2f} / {track_length:.2f} m')
        _, current_idx = track_tree.query([x_current[0], x_current[1]])
        
        idx_diff = current_idx - last_current_idx
        if idx_diff < -len(x_mid)/2: idx_diff += len(x_mid)
        elif idx_diff > len(x_mid)/2: idx_diff -= len(x_mid)
        if idx_diff > 0:
            s_total_traveled += np.sum(segment_lengths[last_current_idx:current_idx])
        last_current_idx = current_idx

        s_projected = s_coor[current_idx]
        x_ref_horizon = np.zeros(n_horizon)
        y_ref_horizon = np.zeros(n_horizon)
        v_ref_horizon = np.zeros(n_horizon)
        current_W_acc = 0.0 if s_total_traveled >= track_length * track_percentual else W_acc

        for k in range(n_horizon):
            s_projected += max(x_current[2], 1.5) * dt 
            s_wrapped = s_projected % track_length
            x_ref_horizon[k] = get_x_at_s(s_wrapped)
            y_ref_horizon[k] = get_y_at_s(s_wrapped)
            
            if s_total_traveled >= track_length * track_percentual:
                v_ref_horizon[k] = 0.0
            #elif turning:
            #    v_ref_horizon[k] = TurningSpeed(np.rad2deg(u_prev[1]), V_cruising, V_turning * 0.8)
            else:
                v_ref_horizon[k] = TurningSpeed(np.rad2deg(u_prev[1]), V_cruising, V_turning, k=0.15)
            
        # Vectorized calculation of linearization systems
        A_mat, B_mat, c_mat = get_linear_model_matrices_all(x_current, u_prev, dt, n_horizon, l)

        if s_total_traveled >= track_length * track_percentual:
            W_acc_param.value = 0.0  # Eliminate the braking penalty completely
        else:
            W_acc_param.value = current_W_acc


        # Assign values directly to the graph parameters (Instantaneous update)
        x_init_param.value = x_current
        u_prev_param.value = u_prev
        x_ref_param.value = x_ref_horizon
        y_ref_param.value = y_ref_horizon
        v_ref_param.value = v_ref_horizon

        for k in range(n_horizon):
            A_params[k].value = A_mat[k]
            B_params[k].value = B_mat[k]
            c_params[k].value = c_mat[k]
        
        try:
            # Native warm_start dramatically improves MOSEK performance over step updates
            # canon_backend handles problem parsing via SciPy to avoid the _cvxcore error
            prob.solve(
                solver=cp.MOSEK, 
                verbose=False, 
                warm_start=True, 
                canon_backend=cp.SCIPY_CANON_BACKEND
            )
            u_control = U_cvx[:, 0].value
            
            if u_control is None: 
                raise ValueError("Solver returned None")
                
        except Exception:
            u_control = np.array([0.0, u_prev[1]])

        beta_sim = np.arctan(0.5 * np.tan(u_control[1]))
        x_next = x_current[0] + dt * (x_current[2] * np.cos(x_current[3] + beta_sim))
        y_next = x_current[1] + dt * (x_current[2] * np.sin(x_current[3] + beta_sim))
        v_next = x_current[2] + dt * u_control[0]
        psi_next = x_current[3] + dt * ((x_current[2] / l) * np.sin(beta_sim))
        
        turning = abs(u_control[1]) >= np.deg2rad(4.5)
        x_current = np.array([x_next, y_next, v_next, psi_next])
        u_prev = u_control.copy()
        
        t_history.append((step + 1) * dt)
        x_history.append(x_current[0])
        y_history.append(x_current[1])
        v_history.append(x_current[2])
        v_ref_history.append(v_ref_horizon[0])
        acc_history.append(u_control[0])
        delta_history.append(np.rad2deg(u_control[1]))
        psi_history.append(np.rad2deg(x_current[3]))
        turning_history.append(1 if turning else 0)
        delta_check = np.abs(np.array(delta_history))

        Vref_mean = abs(np.mean(np.array(v_ref_history[-(int(5/dt)):])))
        V_mean = abs(np.mean(np.array(v_history[-(int(5/dt)):])))
        acc_mean = abs(np.mean(np.array(acc_history[-(int(5/dt)):])))

        #print('X_cvx: ', X_cvx.value)

        if len(delta_check[delta_check > 26.5]) > 10 :
            BreakCheck = True
            if show: print('Angle Break')
            break

        if  s_total_traveled >= track_length * track_percentual*1.1:
            BreakCheck = True
            if show: print('Length Break')
            break

        elif s_total_traveled >= track_length * track_percentual and x_current[2] < 0.75:
            if show: print('Completed')
            break

        elif t_history[-1] > t_lim:
            BreakCheck = True
            if show: print('Time Break')
            break

        elif t_history[-1] > t_lim and acc_mean < 0.1 and V_mean > 0.9:
            BreakCheck = True
            if show: print('Time and Speed Break')
            break
        
        elif Vref_mean < 0.01 and acc_mean < 0.1 and x_current[2] > 0.9:
            BreakCheck = True
            if show: print('Vref and Acc Break')
            break
    
    if show: print('s_total:',s_total_traveled, 'track_length:',track_length)

    t_history = np.array(t_history)
    score = RMSE(np.array(v_ref_history)[t_history < t_lim], np.array(v_history)[t_history < t_lim])
    turning_history.append(1 if turning else 0)

    return score, BreakCheck, [t_history, v_history, acc_history, delta_history, turning_history, x_history, y_history, x_mid, y_mid, psi_history, v_ref_history]


Erro: 3.723294244117132 parameters:  [1.5, 9.5, 29.0, 28.5, 9.5, 18.0, 9.0] - x_current[2] < < 0.35\
Erro: 3.682893403585399 parameters:  [3.0, 9.0, 25.0, 17.5, 4.0, 22.5, 12.5] - x_current[2] < 0.45\
Erro: 3.783163494239727 parameters:  [1.0, 17.5, 15.75, 25.75, 7.75, 22.25, 14.5] - x_current[2] < 0.2

In [ ]:

def objective(trial):
    W_X = trial.suggest_float('W_X', 1, 30,step=0.5)
    W_Y = trial.suggest_float('W_Y', 1, 30,step=0.5)
    W_speed = trial.suggest_float('W_speed', 0.25, 30,step=0.25)
    W_acc = trial.suggest_float('W_acc'    , 0.25, 30,step=0.25)
    W_delta = trial.suggest_float('W_delta', 0.25, 20,step=0.25)
    W_U0 = trial.suggest_float('W_U0'      , 0.25, 30,step=0.25)
    W_U1 = trial.suggest_float('W_U1'      , 0.25, 20,step=0.25)
    
    score,BreakCheck,_ = SimulateRT(dt=0.1, n_horizon=30, sim_steps=500,track_percentual=0.9,
                                    W_X=W_X, W_Y=W_Y, W_speed=W_speed, W_acc=W_acc, W_delta=W_delta,
                                    W_U0=W_U0, W_U1=W_U1, size=1, show=False)
    if BreakCheck:
        raise TrialPruned()
    
    return score

study = optuna.create_study(
    direction='minimize',
    #directions=["minimize", "minimize"],
    sampler=SelSampler(mode='auto'),
    #pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}'
    )

study.optimize(objective, n_trials=10)
best_params = study.best_params
params = list(best_params.values())
print('Erro:', study.best_value, 'parameters: ', params)


In [32]:
if 'params' in globals(): params = params
else: params = [3.0, 3.0, 26.25, 7.0, 12.5, 14.75, 8.25]
W_X,W_Y,W_speed,W_acc,W_delta,W_U0,W_U1 =   [1.0, 17.5, 15.75, 25.75, 7.75, 22.25, 14.5]

score,BreakCheck,data = SimulateRT(dt=0.1, n_horizon=30, sim_steps=900,track_percentual=0.9,
                                    W_X=W_X, W_Y=W_Y, W_speed=W_speed, W_acc=W_acc, W_delta=W_delta,
                                    W_U0=W_U0, W_U1=W_U1, size=1, show=True)
[t_history, v_history, acc_history, delta_history, turning_history, x_history, y_history, x_mid, y_mid, psi_history, v_ref_history] = data

Step: 0 | Speed: 0.00 m/s | Distance traveled: 0.00 / 374.15 m
Step: 50 | Speed: 11.48 m/s | Distance traveled: 33.98 / 374.15 m
Step: 100 | Speed: 8.46 m/s | Distance traveled: 85.47 / 374.15 m
Step: 150 | Speed: 8.04 m/s | Distance traveled: 125.59 / 374.15 m
Step: 200 | Speed: 7.78 m/s | Distance traveled: 165.42 / 374.15 m
Step: 250 | Speed: 11.16 m/s | Distance traveled: 214.81 / 374.15 m
Step: 300 | Speed: 9.31 m/s | Distance traveled: 265.78 / 374.15 m
Step: 350 | Speed: 9.04 m/s | Distance traveled: 310.46 / 374.15 m
Step: 400 | Speed: 2.32 m/s | Distance traveled: 349.00 / 374.15 m
Completed
s_total: 351.69712794813745 track_length: 374.14720911227874


In [33]:
ySeries=[v_ref_history, v_history, acc_history, delta_history][:]
xSeries=[t_history for i in range(len(ySeries))][:]
names = ['Reference Speed (m/s)', 'Current Speed (m/s)',
        'Acceleration (m/s²)', 'Stirring Angle (deg)',][:]
data = [x_mid, y_mid, x_history,y_history, t_history, delta_history, psi_history, v_history]

PlotSeriesPLY(xSeries,ySeries,names,title=f'Parameters Over Time|RMSE: {score}')
#PlotSeriesDifScalesPLY(xSeries,ySeries,names,title=f'Parameters Over Time|RMSE: {score}')
PlotMPCTracksPLY(data,width=800,height=500)